# Aufgabe 2: Hodgkin-Huxley-Gleichungssystem

Euler & RK4 (selbst), Vergleich mit scipy.odeint, Stromvariation, Stromimpuls.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt


## Modell-Setup (Hodgkin-Huxley)

Die folgenden Definitionen (Parameter, Ratenfunktionen und die rechte Seite des DGL-Systems) bilden das Hodgkin-Huxley-Modell. Sie werden in allen folgenden Aufgaben dieses Notebooks verwendet.

In [ ]:
# ==== Setup: Hodgkin-Huxley-Modell (Parameter, Ratenfunktionen, rechte Seite) ====
# Biologische Parameter (Projektbeschreibung, Abschnitt 1.2.2)
V_POT = -65.0      # Ruhepotential [mV]
C = 1.0            # Membrankapazität [uF/cm^2]
U_K = -77.0        # Gleichgewichtspotential Kalium [mV]
U_NA = 50.0        # Gleichgewichtspotential Natrium [mV]
U_L = -54.387      # Gleichgewichtspotential Leck [mV]
G_K = 36.0         # maximale Leitfähigkeit Kalium [mS/cm^2]
G_NA = 120.0       # maximale Leitfähigkeit Natrium [mS/cm^2]
G_L = 0.3          # Leck-Leitfähigkeit [mS/cm^2]

# Ratenfunktionen alpha/beta der Gating-Variablen
def alpha_n(U): return -0.01 * (55.0 + U) / (np.exp(-(55.0 + U) / 10.0) - 1.0)
def beta_n(U):  return 0.125 * np.exp(-(65.0 + U) / 80.0)
def alpha_m(U): return -0.1 * (40.0 + U) / (np.exp(-(40.0 + U) / 10.0) - 1.0)
def beta_m(U):  return 4.0 * np.exp(-(65.0 + U) / 18.0)
def alpha_h(U): return 0.07 * np.exp(-(65.0 + U) / 20.0)
def beta_h(U):  return 1.0 / (np.exp(-(35.0 + U) / 10.0) + 1.0)

def x_infinity(alpha, beta):
    return alpha / (alpha + beta)

def ionic_currents(U, n, m, h):
    i_k = G_K * n ** 4 * (U - U_K)
    i_na = G_NA * m ** 3 * h * (U - U_NA)
    i_l = G_L * (U - U_L)
    return i_k, i_na, i_l

def rhs(state, t, I_ext):
    U, n, m, h = state
    I = I_ext(t) if callable(I_ext) else I_ext
    i_k, i_na, i_l = ionic_currents(U, n, m, h)
    dU = (I - i_k - i_na - i_l) / C
    dn = alpha_n(U) * (1 - n) - beta_n(U) * n
    dm = alpha_m(U) * (1 - m) - beta_m(U) * m
    dh = alpha_h(U) * (1 - h) - beta_h(U) * h
    return np.array([dU, dn, dm, dh])

def initial_state():
    U0 = V_POT
    n0 = x_infinity(alpha_n(U0), beta_n(U0))
    m0 = x_infinity(alpha_m(U0), beta_m(U0))
    h0 = x_infinity(alpha_h(U0), beta_h(U0))
    return np.array([U0, n0, m0, h0])

## 2a) Euler-Verfahren, 50 ms, konstanter Strom $I=I_0

In dieser Zelle wird das explizite Euler-Verfahren als Funktion solve_euler definiert und über 50 ms auf das Hodgkin-Huxley-System angewendet, mit konstantem Grundstrom I_0 = -5 nA. Erwartet wird kein Aktionspotential: Bei diesem leicht hyperpolarisierenden Strom bleibt das Neuron in Ruhe, die Spannung sinkt glatt auf etwa -72 mV und verharrt dort. Ein flacher Verlauf ohne Spike bestätigt, dass Solver und Anfangsbedingungen zusammenpassen.

In [ ]:
# 2a) explizites Euler-Verfahren als Funktion
def solve_euler(rhs_func, y0, t):
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        dt = t[i + 1] - t[i]
        y[i + 1] = y[i] + dt * rhs_func(y[i], t[i])
    return y

t = np.arange(0, 50, 0.01)
f = lambda y, t: rhs(y, t, I_ext=-5.0)
y = solve_euler(f, initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, Euler, I₀ = −5 nA (Ruhezustand)")

## 2b) RK4 Implementierung

Hier wird das klassische Runge-Kutta-Verfahren vierter Ordnung als Funktion solve_rk4 definiert und mit denselben Parametern wie beim Euler-Verfahren angewendet. Das Ergebnis ist praktisch identisch (Ruhezustand, glatter Verlauf), da beide Verfahren bei dieser feinen Schrittweite genau genug sind. RK4 wertet die rechte Seite pro Schritt viermal aus und ist dadurch pro Schritt deutlich genauer als Euler.

In [ ]:
# 2b) klassisches Runge-Kutta-Verfahren (RK4) als Funktion
def solve_rk4(rhs_func, y0, t):
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        dt = t[i + 1] - t[i]
        k1 = dt * rhs_func(y[i], t[i])
        k2 = dt * rhs_func(y[i] + 0.5 * k1, t[i] + 0.5 * dt)
        k3 = dt * rhs_func(y[i] + 0.5 * k2, t[i] + 0.5 * dt)
        k4 = dt * rhs_func(y[i] + k3, t[i] + dt)
        y[i + 1] = y[i] + (k1 + 2 * k2 + 2 * k3 + k4) / 6
    return y

t = np.arange(0, 50, 0.01)
f = lambda y, t: rhs(y, t, I_ext=-5.0)
y = solve_rk4(f, initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, RK4, I₀ = −5 nA (Ruhezustand)")

## 2b) Stabilitätsvergleich Euler vs. RK4

In dieser Zelle werden Euler und RK4 bei einem spikeauslösenden Strom (I_0 = 10 nA) für mehrere Schrittweiten dt verglichen. Steile Flanken fordern den Solver, deshalb wird hier Instabilität sichtbar. Man beobachtet, dass das Euler-Verfahren für dt ab etwa 0.08 ms instabil wird und aus dem Bild läuft, während RK4 bis etwa 0.09 ms stabil bleibt. Erst bei dt = 0.1 ms divergieren beide. RK4 verträgt also ungefähr die doppelte Schrittweite bei gleicher Stabilität.

In [ ]:

I0 = 10.0                          # spikeauslösender Strom -> steile Flanken fordern den Solver
dts = [0.01, 0.05, 0.08, 0.09]

fig, (axE, axR) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for dt in dts:
    t = np.arange(0, 50, dt)
    f = lambda y, t: rhs(y, t, I0)
    UE = solve_euler(f, initial_state(), t)[:, 0]
    UR = solve_rk4(f,   initial_state(), t)[:, 0]
    axE.plot(t, UE, label=f"dt={dt}")
    axR.plot(t, UR, label=f"dt={dt}")

for ax, titel in ((axE, "Euler"), (axR, "RK4")):
    ax.set_title(titel); ax.set_xlabel("t [ms]"); ax.legend()
    ax.set_ylim(-100, 120)         # begrenzen, sonst zerdrückt die explodierende Kurve alles
axE.set_ylabel("U [mV]")
plt.tight_layout()
plt.show()

## 2b) odeint Implementierung

Zum Vergleich wird dasselbe System mit der fertigen Funktion odeint aus scipy gelöst, die als Referenz dient. odeint wählt seine Schrittweite intern adaptiv und wechselt je nach Steifigkeit des Problems automatisch das Verfahren. Dadurch bleibt sie auch dort stabil, wo das explizite Euler-Verfahren bei großem dt versagt.

In [ ]:
from scipy.integrate import odeint

t = np.arange(0, 50, 0.01)
f = lambda y, t: rhs(y, t, I_ext=-5.0)
y = odeint(f, initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, odeint, I₀ = −5 nA (Ruhezustand)")

## 2b) Performance Vergleich

Hier wird die reine Laufzeit der drei Verfahren gemessen. Euler ist pro Schritt am schnellsten, weil er die rechte Seite nur einmal auswertet, RK4 braucht vier Auswertungen und ist entsprechend langsamer. odeint ist trotz adaptiver Schrittweite oft überraschend schnell, weil es kompilierter Code ist. Aussagekräftig ist am Ende die Genauigkeit pro Rechenzeit, denn Euler ist zwar schnell pro Schritt, braucht aber eine kleinere Schrittweite, um stabil zu bleiben.

In [ ]:
import time

def zeit(fn, wiederholungen=10):
    t0 = time.perf_counter()
    for _ in range(wiederholungen):
        fn()
    return (time.perf_counter() - t0) / wiederholungen * 1000  # ms pro Durchlauf

verfahren = {
    "Euler":  lambda: solve_euler(f, initial_state(), t),
    "RK4":    lambda: solve_rk4(f, initial_state(), t),
    "odeint": lambda: odeint(rhs, initial_state(), t, args=(-5.0,)),
}

for name, fn in verfahren.items():
    print(f"{name:8s}: {zeit(fn):.3f} ms")

## 2c) Variation von $I_0$ zwischen -5 nA und 15 nA mit RK4

In dieser Zelle wird der konstante Strom I_0 zwischen -5 und 15 nA variiert und jeweils der Spannungsverlauf geplottet, um das Schwellenverhalten sichtbar zu machen. Die ausführliche Deutung folgt in der Zelle unter dem Plot.

In [ ]:

I_0 = [-5.0, 0.0, 2.2, 5.0, 10.0, 15.0]  # Stromstärken in nA
t = np.arange(0, 50, 0.01)

fig, axes = plt.subplots(len(I_0), 1, figsize=(8, 1.8*len(I_0)), sharex=True)
for ax, I in zip(axes, I_0):
    f = lambda y, t: rhs(y, t, I)          # eigener Solver (RK4)
    U = solve_rk4(f, initial_state(), t)[:, 0]
    ax.plot(t, U)
    ax.set_ylabel("U [mV]")
    ax.set_title(f"I₀ = {I} nA", loc="left", fontsize=10)
    ax.axhline(0, color="gray", lw=0.5, ls="--")   # Orientierung: Spike-Schwelle grob bei 0 mV
axes[-1].set_xlabel("t [ms]")
plt.tight_layout()
plt.show()


Erklärung

Bei Variation des konstanten Stroms $I_0$ zeigt sich ein klares Schwellenverhalten.
Für kleine Ströme ($I_0 \lesssim 2$ nA, insbesondere der Grundstrom $I_0 = -5$ nA)
bleibt das Neuron inaktiv: Die Membranspannung verharrt nahe dem Ruhepotential bzw.
zeigt nur eine kleine unterschwellige Auslenkung, aber kein Aktionspotential.
Der Reiz reicht nicht aus, um die Natriumkanäle ausreichend zu öffnen.

Oberhalb einer Schwelle von etwa $I_0 \approx 2.3$ nA wird ein Aktionspotential
ausgelöst: $U$ steigt sprunghaft auf $\approx +38$ mV an und fällt anschließend unter
das Ruhepotential zurück (Nachhyperpolarisation), bevor es sich wieder erholt. Für
noch größere Ströme feuert das Neuron periodisch, und die Feuerrate steigt mit
$I_0$ (in 50 ms z. B. 1 Spike bei 3 nA, 2 bei 6 nA, 4 bei 10 nA). Dies entspricht der
klassischen Frequenz-Strom-Beziehung: Ein stärkerer Dauerreiz führt nicht zu höheren
Spitzen, sondern zu häufigeren Aktionspotentialen. In den folgenden Aufgaben wird
wieder fest $I_0 = -5$ nA verwendet.

## 2d) Stromimpuls (t=10..11 ms, I_imp=50 nA): U, I, n, m, h

In dieser Zelle wird ein kurzer Stromimpuls eingebaut und U, I, n, m und h gemeinsam dargestellt. So wird sichtbar, wie ein einzelnes Aktionspotential entsteht und welche Rolle die Gatingvariablen dabei spielen. Die ausführliche Deutung folgt in der Zelle unter dem Plot.

In [ ]:
def stromimpuls(t):
    if 10 <= t <= 11:
        return 50.0
    else:
        return -5.0

t = np.arange(0, 50, 0.01)
f = lambda y, t: rhs(y, t, stromimpuls(t))
y = solve_rk4(f, initial_state(), t)
U = y[:, 0]
I = np.array([stromimpuls(ti) for ti in t])
n = y[:, 1]  # Na⁺-Kanäle
m = y[:, 2]  # K⁺-Kanäle
h = y[:, 3]  # Inaktivierung Na⁺-Kanäle

groessen = [(U, "U [mV]"), (I, "I [nA]"), (n, "n"), (m, "m"), (h, "h")]

fig, axes = plt.subplots(5, 1, figsize=(8, 9), sharex=True)
for ax, (daten, label) in zip(axes, groessen):
    ax.plot(t, daten)
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("t [ms]")
axes[0].set_title("2d) Stromimpuls: I₀ = −5 nA, Impuls 50 nA bei t = 10–11 ms", loc="left")
plt.tight_layout()
plt.show()

2d) Stromimpuls und Funktion der Gatingvariablen

Ausgehend vom Ruhezustand ($I_0 = -5$ nA) wird für $t \in [10, 11]$ ms ein kurzer,
starker Impuls von $I_\mathrm{imp} = 50$ nA angelegt. Dieser depolarisiert die Membran
so weit, dass ein einzelnes Aktionspotential ausgelöst wird. Der zeitliche Ablauf
lässt sich vollständig über die drei Gatingvariablen verstehen, die auf sehr
unterschiedlichen Zeitskalen reagieren:

- $m$ ($\mathrm{Na}^+$-Aktivierung) hat die schnellste Zeitkonstante. Bei der
  Depolarisation öffnet $m$ fast augenblicklich, die Natriumkanäle leiten, und der
  einströmende $\mathrm{Na}^+$-Strom treibt $U$ steil nach oben (Aufstrich des Spikes
  bis $\approx +40$ mV), ein selbstverstärkender Prozess.
- $h$ ($\mathrm{Na}^+$-Inaktivierung) reagiert langsamer und fällt während des
  Spikes ab. Dadurch werden die Natriumkanäle wieder geschlossen und der
  $\mathrm{Na}^+$-Einstrom gestoppt.
- $n$ ($\mathrm{K}^+$-Aktivierung) steigt ebenfalls verzögert an, öffnet die
  Kaliumkanäle, und der ausströmende $\mathrm{K}^+$-Strom repolarisiert die Membran, $U$ fällt wieder ab.

Das Zusammenspiel „schnelles $m$ gegen langsames $h$ und $n$" erklärt die Form des
Aktionspotentials: schneller Anstieg durch $\mathrm{Na}^+$, anschließende Repolarisation
durch $\mathrm{K}^+$. Da $n$ nach dem Spike noch erhöht und $h$ noch niedrig ist,
unterschreitet $U$ kurzzeitig das Ruhepotential. 